# orbis-3d: Colab training run

Thin wrapper around the `orbis` CLI. The same code path runs locally, so anything that
works here works from a terminal with a CUDA GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

Put the walkaround video in Google Drive first. This notebook copies it into the repo,
runs the pipeline, and downloads a share bundle you can open in a browser.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/GordenArcher/orbis-3d.git"
PROJECT_DIR = "/content/orbis-3d"

# Point this at the video in Drive. Portrait or landscape both work.
CAPTURE_SOURCE = "/content/drive/MyDrive/orbis/capture/chair.mp4"

# One folder per capture under processing/. Reusing a name resumes that workspace.
WORKSPACE_NAME = "chair"

# 300 frames keeps COLMAP work reasonable on a free GPU. Raise it only if the
# camera track looks unstable or the splat is missing detail.
FRAMES_TARGET = 300
MAX_ITERATIONS = 30000

## 2. Mount Drive

Skip this cell if the capture is not on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Get the repo

In [ ]:
import os
import subprocess

if os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(os.getcwd())

## 4. Install

Colab already ships a CUDA build of torch. `scripts/colab_setup.sh` installs ffmpeg,
ninja, and Nerfstudio without replacing torch. The first build of tinycudann takes
roughly 5 to 10 minutes.

In [ ]:
!bash scripts/colab_setup.sh

## 5. Copy the capture into the repo

In [ ]:
import shutil
from pathlib import Path

source = Path(CAPTURE_SOURCE)
destination = Path("capture") / source.name
shutil.copy(source, destination)
print(destination)

## 6. Check the capture

Fails on anything that would waste the GPU run: too short, too low resolution, or not a
readable video.

In [ ]:
import subprocess

subprocess.run(["orbis", "check", str(destination)], check=True)

## 7. Run the pipeline

COLMAP, splatfacto training, PLY export, then packaging. A 300 frame person scan is
usually 10 to 25 minutes on a T4.

In [ ]:
subprocess.run(
    [
        "orbis",
        "run",
        str(destination),
        "--name",
        WORKSPACE_NAME,
        "--frames-target",
        str(FRAMES_TARGET),
        "--max-iterations",
        str(MAX_ITERATIONS),
    ],
    check=True,
)

## 8. Download the bundle

Unzip it, run `python -m http.server 8000` inside the folder, and open
http://localhost:8000. Dragging `scene.ply` onto `index.html` also works.

In [ ]:
from google.colab import files

share_zip = Path("processing") / WORKSPACE_NAME / f"{WORKSPACE_NAME}-share.zip"
files.download(str(share_zip))